# Previsão de incidentes por turno

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import (GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor)

In [18]:
df = pd.read_excel("LW-DATASET-TRATADO.xlsx")
print(df.shape)

(122543, 28)


### Tratamento e Filtragem Inicial dos Dados

In [19]:
df['Aberto'] = pd.to_datetime(df['Aberto'], errors='coerce')
df = df.loc[df['Aberto'].notna() & (df['Aberto'] >= '2025-01-01')].copy()

``Construção da Série Temporal de Incidentes``

Nesta etapa, cada incidente é associado a uma data e a um turno operacional com base no horário de abertura.

incidentes são agrupados por dia e turno, gerando a quantidade total de incidentes observados em cada combinação.

In [ ]:
df['data'] = df['Aberto'].dt.normalize()

df['turno'] = pd.cut(
    df['Aberto'].dt.hour, bins=[0, 6, 12, 18, 24],
    labels=['Madrugada', 'Manhã', 'Tarde', 'Noite'], right=False,
    include_lowest=True)

serie = (df.groupby(['data', 'turno'], observed=False).size().reset_index(name='qtd_incidentes'))
display(serie.head())

Total de registros: 121,811
Período analisado: 2025-01-01 a 2025-12-31


,data,turno,qtd_incidentes
0,2025-01-01,Madrugada,18
1,2025-01-01,Manhã,5
2,2025-01-01,Tarde,4
3,2025-01-01,Noite,7
4,2025-01-02,Madrugada,8


``Completação da Série Temporal``

Quando não existe registro de incidentes para determinado dia e turno, o valor é preenchido com zero.

In [ ]:
datas = pd.date_range(serie['data'].min(), serie['data'].max(), freq='D')

idx = pd.MultiIndex.from_product([datas, ['Madrugada', 'Manhã', 'Tarde', 'Noite']], names=['data', 'turno'])

serie = (serie.set_index(['data', 'turno']).reindex(idx, fill_value=0).reset_index())

print(f'Total de registros: {len(df):,}')
print(f'Período analisado: {datas.min():%Y-%m-%d} a {datas.max():%Y-%m-%d}')

### Engenharia de Features

- `turno_code`: representação numérica do turno operacional
- `dow`: dia da semana
- `mes`: mês 
- `is_weekend`: flag fim de semana
- `lag_1`: qtd de incidentes no mesmo turno do dia anterior
- `lag_7`: qtd de incidentes no mesmo turno há 7 dias
- `lag_14`: qtd de incidentes no mesmo turno há 14 dias
- `media_7d`: média de incidentes dos últimos 7 dias para o mesmo turno
- `media_14d`: média de incidentes dos últimos 14 dias para o mesmo turno

In [21]:
serie['turno_code'] = serie['turno'].map({'Madrugada': 0, 'Manhã': 1, 'Tarde': 2, 'Noite': 3})

serie['dow'] = serie['data'].dt.dayofweek
serie['mes'] = serie['data'].dt.month
serie['is_weekend'] = (serie['dow'] >= 5).astype(int)

g = serie.groupby('turno', observed=False)['qtd_incidentes']

serie['lag_1'] = g.shift(1)
serie['lag_7'] = g.shift(7)
serie['lag_14'] = g.shift(14)

serie['media_7d'] = g.transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
serie['media_14d'] = g.transform(lambda s: s.shift(1).rolling(14, min_periods=1).mean())

``Seleção das Features``

- **features_base**: variáveis temporais e histórico recente de incidentes
- **features_plus**: conjunto maior contendo informações adicionais de tendência observadas nos últimos 14 dias

In [22]:
features_base = ['lag_1', 'lag_7', 'media_7d', 'mes', 'dow', 'is_weekend']
features_plus = features_base + ['media_14d', 'lag_14']

serie_model = serie.dropna(subset=features_plus).reset_index(drop=True)
print(f'Observações modeláveis: {len(serie_model):,}')

Observações modeláveis: 1,404


### Métricas Avaliativas

In [23]:
def safe_mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.any() else np.nan

def metricas(y_true, y_pred):
    return {
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'r2': r2_score(y_true, y_pred),
        'mape': safe_mape(y_true, y_pred)}

### Split Temporal

In [24]:
split = int(len(serie_model) * 0.8)
train, test = serie_model.iloc[:split], serie_model.iloc[split:]
print(f'Treino: {len(train):,} | Teste: {len(test):,} | corte na linha: {split}')

Treino: 1,123 | Teste: 281 | corte na linha: 1123


### Benchmark de Modelos

In [25]:
models = {
    'GradientBoosting_base': (
        GradientBoostingRegressor(
            n_estimators=200, max_depth=2,
            learning_rate=0.03, loss='huber',
            random_state=42), features_base),

    'GradientBoosting_plus': (
        GradientBoostingRegressor(
            n_estimators=200, max_depth=2,
            learning_rate=0.03, loss='huber', 
            random_state=42), features_plus),

    'RandomForest_plus': (
        RandomForestRegressor(
            n_estimators=300, max_depth=8,
            min_samples_leaf=2, random_state=42,
            n_jobs=-1), features_plus),

    'ExtraTrees_plus': (
        ExtraTreesRegressor(
            n_estimators=300, max_depth=8,
            min_samples_leaf=2, random_state=42,
            n_jobs=-1 ), features_plus),
            
    'HistGradientBoosting_plus': (
        HistGradientBoostingRegressor(
            max_iter=200, max_leaf_nodes=15,
            learning_rate=0.04, l2_regularization=1.0,
            random_state=42 ), features_plus)}

In [26]:
resultados = []
predicoes = {}

for nome, (modelo, features) in models.items():
    X_train = train[features + ['turno_code']]
    X_test = test[features + ['turno_code']]
    y_train = train['qtd_incidentes']
    y_test = test['qtd_incidentes']

    modelo.fit(X_train, y_train)
    previsoes = np.maximum(0, modelo.predict(X_test))

    predicoes[nome] = (test[['data', 'turno', 'qtd_incidentes']]
        .assign(numero_previsto=previsoes, modelo=nome))

    resultados.append({'modelo': nome, **metricas(y_test, previsoes)})

In [ ]:
# baseline - repete o valor do mesmo turno no dia anterior
baseline = np.maximum(0, test['lag_1'])

resultados.append({'modelo': 'Baseline_lag_1', **metricas(y_test, baseline)})
comparacao = (pd.DataFrame(resultados).sort_values(['mae', 'rmse']).reset_index(drop=True))
display(comparacao.style.format({'mae': '{:.2f}', 'rmse': '{:.2f}', 'r2': '{:.4f}', 'mape': '{:.2f}%'}))

,modelo,mae,rmse,r2,mape
0,GradientBoosting_base,49.25,82.44,0.0235,22.32%
1,GradientBoosting_plus,49.39,83.09,0.0082,22.23%
2,RandomForest_plus,51.88,81.86,0.0372,25.45%
3,ExtraTrees_plus,54.48,82.04,0.0330,26.93%
4,HistGradientBoosting_plus,59.08,88.06,-0.1139,29.37%
5,Baseline_lag_1,61.19,102.81,-0.5184,31.59%
6,Baseline_lag_1,61.19,102.81,-0.5184,31.59%


In [ ]:
best_model = comparacao.iloc[0]['modelo']
best_pred = predicoes[best_model].copy()

print(f'Melhor modelo: {best_model}')
print('Métricas gerais:',{k: round(v, 4)
     for k, v in metricas(best_pred['qtd_incidentes'], best_pred['numero_previsto']).items()})

Melhor modelo: GradientBoosting_base
Métricas gerais: {'mae': 49.2513, 'rmse': np.float64(82.4444), 'r2': 0.0235, 'mape': np.float64(22.3238)}


In [ ]:
metricas_turno = (best_pred
    .groupby('turno', observed=False)
    .apply(lambda df: pd.Series(metricas(df['qtd_incidentes'], df['numero_previsto'])))
    .reset_index()
    [['turno', 'mae', 'mape', 'rmse']].sort_values('turno'))

display(metricas_turno.style.format({'mae': '{:.2f}', 'mape': '{:.2f}%', 'rmse': '{:.2f}'}))

C:\Users\IsabellaHeder\AppData\Local\Temp\ipykernel_35144\1838122887.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: pd.Series(metricas(


,turno,mae,mape,rmse
0,Madrugada,56.65,24.13%,83.36
1,Manhã,59.62,22.34%,117.22
2,Noite,41.98,21.17%,60.75
3,Tarde,38.87,21.68%,53.42


In [31]:
df_previsoes_turno = pd.DataFrame({
    'Data': best_pred['data'].dt.strftime('%Y-%m-%d'),
    'Turno': best_pred['turno'],
    'Incidentes_Previstos': best_pred['numero_previsto'].round(0).astype(int),
    'Incidentes_Reais': best_pred['qtd_incidentes'].astype(int)})

df_previsoes_turno = df_previsoes_turno.sort_values(['Data', 'Turno'])

display(df_previsoes_turno.head())
df_previsoes_turno.to_csv('previsao_incidentes_turno.csv', index=False)

,Data,Turno,Incidentes_Previstos,Incidentes_Reais
1123,2025-10-22,Noite,181,239
1124,2025-10-23,Madrugada,165,172
1125,2025-10-23,Manhã,219,170
1127,2025-10-23,Noite,208,166
1126,2025-10-23,Tarde,198,248
